# Optimizing Agentic AI Collaboration v1.1.0 - GSM8K Full Workflow Template

> **Historical execution.** This notebook records an executed v1.1.x setup run. It is retained for provenance, but its imports and statistics result keys target an earlier `utils` API and it is not expected to run unchanged against the current repository. Use the v1.2.x notebooks or later experiment notebooks as current references.

## Project aims

 1. **a LangChain-based multi-agent Society-of-Minds prototype** supporting **≥3 interaction protocols**,
 2. **benchmark gains** (≥5% on **GSM8K** and **MMLU** vs a single-agent SLM baseline), and
 3. a **cost/benefit profile** using **latency + total token consumption**, all tracked in **W&B** for a claim-to-evidence table.  

## Current notebook version achievements

The code acts as a **template** for execution and evaluation of both single agent baselines and multi-agentic systems over the **GSM8K** dataset. The run is orchestrated and stored through W&B.
It covers the full workflow of the project experiments:
1. Automated ingestion of GSM8K dataset and creation of single/multi agent pipeline.
2. Evaluation of the agentic system over multiple GSM8K examples. Each sample is evaluated with 3 different seeds to reduce estimation variability.
3. Definition of statistically consistent estimators and confidence intervals for accuracy and execution times. The results and the statistical measures are stored on W&B.

However, it does **not yet produce evidence** for the thesis’ central claims (benchmark gains + compute trade-off on GSM8K/MMLU).

## Relation with the Project WPs

**WP2 - T1: End-to-end pipeline template definition.** It establishes a first version of the operational end-to-end framework (load → generate → evaluate → W&B-log) that all WP2 experiment notebooks will use as a template.

## Setup

**Hardware target:** Google Colab T4 GPU (free tier).
**Model:** `mistralai/Mistral-7B-Instruct-v0.2` with 4-bit NF4 quantization.

---
### Additional information

1. **LangChain-based multi-agent orchestration** — agents are defined as LangChain `LLMChain` instances with structured `PromptTemplate`s, composed via a custom sequential pipeline. This replaces ad-hoc string concatenation with a modular, extensible framework.
2. **Weights & Biases (W&B) experiment tracking** — every agent call, its latency, token counts, and full text are logged as W&B artifacts and metrics, enabling richer post-hoc analysis across runs.

---

### Before running

Complete these manual steps in Colab:

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets


## What the current notebook implements

The implementation is a **LangChain refactor** of a *single* 3-step pipeline:

* **Proposer → Critic → Refiner**, plus a **single-agent baseline**.
* **W&B logging** of per-step latency and output length; plus a W&B `Table` comparing outputs across tasks.
* Evaluation is on **MMLU random samples.**
**Observed latencies and overhead (from the notebook run outputs)**. **Accuracy and latency averages are estimated with both a point and a 95% confidence interval**
* Multiple random seeds are defined to **allow reproductibility**.


---

## Objective-by-objective diagnosis

### OBJ-1 (multi-agent prototype, ≥3 protocols)

* **Partially relevant**: there is a working multi-agent scaffold (roles, orchestration, logging).
* **Not compliant** with the “≥3 distinct interaction protocols” requirement: there is **only one protocol** (Critique→Refine). No majority voting, no self-consistency, no multi-round debate.

### OBJ-2 (≥5% accuracy gain on GSM8K + MMLU vs single-agent baseline)

* **Adressed**: dataset harness, scoring based on accuracy (with both point and interval-based estimations) compared to a single-agent baseline. Execution repeated with 3 distinct seeds to gain consistency and reduce estimations variability.

### OBJ-3 (accuracy-to-compute trade-off: latency + total token consumption)

* **Partially addressed**: latency is measured and logged. Average latency is estimated using both a point and confidence intervals.
* **Missing**: “total token consumption” (you log output chars, not tokens), and you’re not logging GPU memory / throughput in a benchmark setting.

### Scope alignment

* Model is **7B** and 4-bit quantized (within the “1B–7B” envelope), so *model class is in-scope*.
* The current artifact is able to compare with statistically consistent measures of the latency and the accuracy the model's execution. It logs all the executions through WandB
* However, the model used remains quite far away from the current state-of-the-art. Furthermore, only 20 distinct GSM8K samples are evaluated, *not enough for a consistent, low-variance estimation of the agents performance*.

---



## 1. Setup and Installation

We install all the libraries required for the Github repo.
For loading models and data, orchestrating agents, and saving the results, we use Hugging Face stack (same as v0.4), plus `langchain`, `langchain-community`, `langchain-core`, `datasets` and `wandb`.
If we are running from Colab, we import the Github repo, which includes utils python files with helper methods.

In [ ]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"   
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

In [2]:
# Install the project dependencies in the selected runtime
%cd {repo_root}
%pip install -r requirements.txt

/kaggle/working/tfg-agentic-ai
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: langchain-core
    Found existing installation: la

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.


In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: kaggle
✅ Secrets loaded from Kaggle Secrets.


## 3. Initialize Weights & Biases

We create a W&B run that will persist for the entire notebook session.  All agent calls, latencies, and outputs will be logged here.


In [4]:
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics

project = "agentic-collab-setup" if (ENVIRONMENT == "colab" or ENVIRONMENT == "kaggle") else "dev-agentic-collab-setup"
print(f"W&B Project: {project}")

run = init_wandb(
    run_name="GSM8K-Evaluation-Mistral-7B-Instruct",
    project=project,
    config={
        "model_id":       "mistralai/Mistral-7B-Instruct-v0.2",
        "quantization":   "nf4-4bit",
        "framework":      "langchain",
        "agents":         ["proposer", "critic", "refiner"],
        "max_new_tokens": {"proposer": 1024, "critic": 1024, "refiner": 1024, "baseline": 1024},
        "temperature":    0.7,
        "top_p":          0.95,
        "top_k":          50,
        "eval_dataset":   "gsm8k",
        "eval_size":      20,
        "data_seed":    42,
        "generation_seeds": [42, 123, 456],
    },
)

define_wandb_chart_only_metrics()
print(f"✅ W&B run: {run.name}  —  {run.url}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


W&B Project: agentic-collab-setup


wandb: Currently logged in as: andreu-fernandez-cubi (tfg-optimizing-agentic-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260602_084215-ycuonn19
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run GSM8K-Evaluation-Mistral-7B-Instruct
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-collab-setup
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-collab-setup/runs/ycuonn19


✅ W&B run: GSM8K-Evaluation-Mistral-7B-Instruct  —  https://wandb.ai/tfg-optimizing-agentic-ai/agentic-collab-setup/runs/ycuonn19


## 4. Data Loading
We load the GSM8K (Grade School Math tasks) dataset, which will be used later to evaluate the agentic workflows.


In [5]:
from utils.data import load_gsm8k

# ── GSM8K DATASET ──────────────────────────────────────────────────
gsm8k_dataset = load_gsm8k(split="test")

print(f"✅ GSM8K dataset loaded with {len(gsm8k_dataset)} examples.")
print(f"Example question: {gsm8k_dataset[0]['question']}")
print(f"Example answer:  {gsm8k_dataset[0]['answer']}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

✅ GSM8K dataset loaded with 1319 examples.
Example question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Example answer:  Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


## 5. Model Loading (4-bit Quantized)

Identical to v0.4: we load `Mistral-7B-Instruct-v0.2` in NF4 4-bit via `bitsandbytes`, which keeps VRAM usage under ~6 GB on a T4.


In [6]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = wandb.config.model_id
model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)

print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

✅ Model loaded on cuda:0 — dtype torch.bfloat16


## 6. LangChain LLM Wrapper

We wrap the local HF model in a `HuggingFacePipeline` so it can be used as the backend for LangChain chains.  This gives us `PromptTemplate` composition, chain sequencing, and callback hooks — all of which we exploit below.


In [7]:
from utils.agents import build_hf_langchain_llm

try:
  assert "generation_seeds" in wandb.config and type(wandb.config.generation_seeds) == int
  generation_seed = wandb.config.generation_seeds
except:
  generation_seed = None
# Wrap the loaded model as a LangChain-compatible LLM
llm = build_hf_langchain_llm(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=wandb.config.max_new_tokens["proposer"],
    temperature=wandb.config.temperature,
    top_k=wandb.config.top_k,
    top_p=wandb.config.top_p,
    return_full_text=False,
    seed=generation_seed,
)

# Quick sanity check
print(llm.invoke("What is the capital of France?")[:200])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_k', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




The capital city of France is Paris. Paris is one of the most famous and visited cities in the world, and it is home to many iconic landmarks such as the Eiffel Tower, the Louvre Museum, Notre-Dame 


## 7. Defining Agents as LangChain Chains

Each agent is a `LLMChain` built from a `PromptTemplate`.  This is a clean separation of *role definition* (the template) from *execution engine* (the LLM), making it trivial to swap models, add agents, or modify prompts independently.


In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Parser
parser = StrOutputParser()

# ── Proposer ──────────────────────────────────────────────────
proposer_template = PromptTemplate.from_template(

        "You are the Proposer Agent. Your role is to analyze the "
        "following task and provide a clear, step-by-step solution. "
        "Be detailed in your reasoning.\n\n"
        "Task: {task}\n\n"
        "Provide your proposed solution below:"
)
proposer_chain = proposer_template | llm | parser

# ── Critic ────────────────────────────────────────────────────
critic_template = PromptTemplate.from_template(
        "You are the Critic Agent. Your role is to meticulously "
        "review the proposed solution for the given task. Identify "
        "any potential flaws, logical errors, missed assumptions, "
        "or areas for improvement. Do not provide a new solution, "
        "only critique the existing one.\n\n"
        "Original Task: {task}\n\n"
        "Proposed Solution to review:\n---\n{proposal}\n---\n\n"
        "Provide your critical analysis below:"
)
critic_chain = critic_template | llm | parser

# ── Refiner ───────────────────────────────────────────────────
refiner_template = PromptTemplate.from_template(
        "You are the Refiner Agent. Your role is to produce a final, "
        "correct solution to a task using an initial proposal and a "
        "critique of that proposal. Carefully consider the critique "
        "and use it to fix the flaws in the original proposal.\n\n"
        "Original Task: {task}\n\n"
        "Proposed Solution:\n---\n{proposal}\n---\n\n"
        "Critique of the Solution:\n---\n{critique}\n---\n\n"
        "Based on the critique, provide the final, corrected solution "
        "below. Explain your steps clearly."
)
refiner_chain = refiner_template | llm | parser

# ── Single-Agent Baseline ────────────────────────────────────
baseline_template = PromptTemplate.from_template(
        "You are an AI assistant. Solve the following task, "
        "providing your final answer and reasoning.\n\n"
        "Task: {task} \n\n"
)
baseline_chain = baseline_template | llm | parser

print("✅ All agent chains defined.")

✅ All agent chains defined.


## 8. Multi-Agent Orchestrator with W&B Logging

The `run_multi_agent` function sequences the three chains and logs per-step metrics to W&B.  It also captures a W&B `Table` for side-by-side comparison.

Key metrics logged per step:
- `latency_s` — wall-clock time
- `output_chars` — proxy for token count
- Full text of each agent's output (as a W&B `Table` row)


In [9]:
import time
from utils.logging_utils import log_metrics

def run_agent_step(chain, inputs, step_name):
    """Run a single LangChain chain, time it, and log to W&B."""
    t0 = time.perf_counter()
    # With LCEL, chain.invoke returns a string directly
    output_text = chain.invoke(inputs)
    elapsed = time.perf_counter() - t0

    # Log scalar metrics
    log_metrics({
        f"{step_name}/latency_s":    round(elapsed, 2),
        f"{step_name}/output_chars": len(output_text),
    })

    print(f"  ⏱  {step_name}: {elapsed:.1f}s  |  {len(output_text)} chars")
    return output_text, elapsed


def run_multi_agent(task, task_label="task", eval_type=None):
    """
    Execute the full Proposer → Critic → Refiner pipeline
    plus a single-agent baseline, with W&B logging.
    """
    print(f"\n{'='*60}")
    print(f"🤖 MULTI-AGENT PIPELINE  —  {task_label}")
    print(f"{'='*60}")

    # Step 1 — Proposer
    print("\n[Step 1] Proposer Agent")
    proposal, t_prop = run_agent_step(
        proposer_chain, {"task": task}, f"{task_label}/proposer"
    )
    if eval_type is None:  # When evaluating on a dataset, the output is not printed to avoid clutter in the Notebook
      print(f"\n[Proposer Output]:\n{proposal[:500]}\n{'...' if len(proposal)>500 else ''}")

    # Step 2 — Critic
    print("\n[Step 2] Critic Agent")
    critique, t_crit = run_agent_step(
        critic_chain, {"task": task, "proposal": proposal}, f"{task_label}/critic"
    )
    if eval_type is None:
      print(f"\n[Critic Output]:\n{critique[:500]}\n{'...' if len(critique)>500 else ''}")

    # Step 3 — Refiner
    print("\n[Step 3] Refiner Agent")
    refined, t_ref = run_agent_step(
        refiner_chain,
        {"task": task, "proposal": proposal, "critique": critique},
        f"{task_label}/refiner",
    )
    if eval_type is None:
      print(f"\n[Refiner Output]:\n{refined[:500]}\n{'...' if len(refined)>500 else ''}")

    # Baseline
    print(f"\n{'='*60}")
    print(f"📊 SINGLE-AGENT BASELINE  —  {task_label}")
    print(f"{'='*60}")
    baseline, t_base = run_agent_step(
        baseline_chain, {"task": task}, f"{task_label}/baseline"
    )
    if eval_type is None:
      print(f"\n[Baseline Output]:\n{baseline[:500]}\n{'...' if len(baseline)>500 else ''}")

    # Log total pipeline latency
    total_pipeline = t_prop + t_crit + t_ref
    log_metrics({
        f"{task_label}/total_pipeline_latency_s": round(total_pipeline, 2),
        f"{task_label}/baseline_latency_s":       round(t_base, 2),
        f"{task_label}/pipeline_overhead_ratio":   round(total_pipeline / max(t_base, 0.01), 2),
    })

    return {
        "proposal": proposal, "critique": critique,
        "refined": refined,   "baseline": baseline,
        "t_prop": t_prop, "t_crit": t_crit,
        "t_ref": t_ref, "t_base": t_base
    }

print("✅ Orchestrator ready.")

✅ Orchestrator ready.


## 9. GSM8K Evaluation


In [10]:
from utils.data import extract_gsm8k_answer
from utils.evaluation import extract_gsm8k_prediction

In [11]:
import random
import numpy as np
from tqdm.notebook import tqdm
from utils.logging_utils import log_metrics
from utils.agents import build_hf_langchain_llm
from utils.evaluation import gsm8k_match
from langchain_core.output_parsers import StrOutputParser

generation_seeds = wandb.config.generation_seeds
sample_size      = wandb.config.eval_size

# Fix the evaluation subset once — all seed runs evaluate the same questions
random.seed(wandb.config.data_seed)
test_sample = random.sample(list(gsm8k_dataset), sample_size)

# ── Correctness & time matrices: rows = problems, columns = seeds ──
agent_correctness_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_correctness_matrix = np.zeros((sample_size, len(generation_seeds)), dtype=int)
agent_time_matrix           = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_time_matrix        = np.zeros((sample_size, len(generation_seeds)), dtype=float)

for run_idx, gen_seed in enumerate(generation_seeds):
    print(f"\n{'='*60}")
    print(f"  Seed run {run_idx + 1}/{len(generation_seeds)}  —  generation_seed={gen_seed}")
    print(f"{'='*60}")

    # Rebuild LLM and all agent chains with the current generation seed
    llm = build_hf_langchain_llm(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=wandb.config.max_new_tokens["proposer"],
        temperature=wandb.config.temperature,
        top_k=wandb.config.top_k,
        top_p=wandb.config.top_p,
        return_full_text=False,
        seed=gen_seed,
    )
    parser = StrOutputParser()
    proposer_chain = proposer_template | llm | parser
    critic_chain   = critic_template   | llm | parser
    refiner_chain  = refiner_template  | llm | parser
    baseline_chain = baseline_template | llm | parser

    results_table = wandb.Table(
        columns=["task", "proposer_output", "critic_output", "refiner_output",
                 "baseline_output", "ground_truth", "refiner_final_output", "baseline_final_output",
                 "proposer_latency", "critic_latency", "refiner_latency", "baseline_latency"]
    )

    for item_idx, item in enumerate(tqdm(test_sample, desc=f"GSM8K seed={gen_seed} ({sample_size} samples)")):
        task = (
            f"{item['question']}\n\n"
            "IMPORTANT: Provide the final numerical answer at the very end "
            "formatted exactly as '#### [number]'."
        )
        ground_truth_full = item['answer']
        true_answer       = extract_gsm8k_answer(ground_truth_full)

        output = None
        try:
            output            = run_multi_agent(task, task_label='gsm8k_sample', eval_type='gsm8k')
            agent_response    = output['refined']
            agent_answer      = extract_gsm8k_prediction(agent_response)
            baseline_response = output['baseline']
            baseline_answer   = extract_gsm8k_prediction(baseline_response)
        except Exception as e:
            print(f"Pipeline failed: {e}")
            agent_answer = baseline_answer = None
            agent_response = baseline_response = str(e)
            if output is None:
                output = {'proposal': '', 'critique': '', 'refined': agent_response,
                          'baseline': baseline_response,
                          't_prop': 0.0, 't_crit': 0.0, 't_ref': 0.0, 't_base': 0.0}

        agent_correct    = gsm8k_match(agent_answer   , true_answer)
        baseline_correct = gsm8k_match(baseline_answer, true_answer)

        # Store in matrices
        agent_correctness_matrix[item_idx, run_idx]    = int(agent_correct)
        baseline_correctness_matrix[item_idx, run_idx] = int(baseline_correct)
        agent_time_matrix[item_idx, run_idx]           = output['t_prop'] + output['t_crit'] + output['t_ref']
        baseline_time_matrix[item_idx, run_idx]        = output['t_base']

        log_metrics({
            "gsm8k_sample/agents_accuracy":   int(agent_correct),
            "gsm8k_sample/baseline_accuracy": int(baseline_correct),
        })

        results_table.add_data(
            task, output['proposal'][:2000], output['critique'][:2000],
            output['refined'][:2000],  output['baseline'][:2000],
            true_answer, agent_answer, baseline_answer,
            round(output['t_prop'], 2), round(output['t_crit'], 2),
            round(output['t_ref'],  2), round(output['t_base'], 2),
        )

    # Log results table per seed
    wandb.log({f"seed_{gen_seed}/results_comparison": results_table})



  Seed run 1/3  —  generation_seed=42


GSM8K seed=42 (20 samples):   0%|          | 0/20 [00:00<?, ?it/s]

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤖 MULTI-AGENT PIPELINE  —  gsm8k_sample

[Step 1] Proposer Agent


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  ⏱  gsm8k_sample/proposer: 14.6s  |  446 chars

[Step 2] Critic Agent


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  gsm8k_sample/critic: 21.8s  |  1140 chars

[Step 3] Refiner Agent


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  gsm8k_sample/refiner: 17.8s  |  574 chars

📊 SINGLE-AGENT BASELINE  —  gsm8k_sample
  ⏱  gsm8k_sample/baseline: 15.1s  |  494 chars


## 10. Log Results Table & Finish W&B Run

Push the accumulated comparison table to W&B and close the run.  You can view the full dashboard (latency charts, text comparisons, overhead ratios) at the URL printed when the run was initialized.


In [12]:
import tempfile
from utils.logging_utils import finish_wandb
from utils.statistics import (
    wilson_ci_clustered,
    time_statistics,
    welch_t_test_times,
    print_accuracy_report,
    print_time_report,
    print_welch_report,
)

# ── Agent accuracy statistics ─────────────────────────────────
agent_acc_stats = wilson_ci_clustered(agent_correctness_matrix)
print_accuracy_report("Agent Accuracy (GSM8K)", agent_acc_stats)

# ── Baseline accuracy statistics ──────────────────────────────
baseline_acc_stats = wilson_ci_clustered(baseline_correctness_matrix)
print_accuracy_report("Baseline Accuracy (GSM8K)", baseline_acc_stats)

# ── Agent execution time statistics ───────────────────────────
agent_time_stats = time_statistics(agent_time_matrix)
print_time_report("Agent Execution Time (GSM8K)", agent_time_stats)

# ── Baseline execution time statistics ────────────────────────
baseline_time_stats = time_statistics(baseline_time_matrix)
print_time_report("Baseline Execution Time (GSM8K)", baseline_time_stats)

# # ── Time comparison: Welch t-test ─────────────────────────────
# time_comparison = welch_t_test_times(
#     agent_time_stats["per_problem_means"],
#     baseline_time_stats["per_problem_means"],
# )
# print_welch_report("Agent vs Baseline Time (GSM8K)", time_comparison)

# ── Log to W&B summary ────────────────────────────────────────
wandb.summary["n_seed_runs"]                 = len(generation_seeds)
wandb.summary["total_experiments"]           = sample_size * len(generation_seeds)
wandb.summary["avg_pipeline_overhead_ratio"] = round(
    float(np.sum(agent_time_matrix)) / max(float(np.sum(baseline_time_matrix)), 0.01), 2
)

# Agents accuracy
wandb.summary["agents/accuracy"]       = round(agent_acc_stats["p_hat"], 6)
wandb.summary["agents/accuracy_ci"]    = [round(agent_acc_stats["ci_lower"], 6),
                                          round(agent_acc_stats["ci_upper"], 6)]
wandb.summary["agents/accuracy_se"]    = round(agent_acc_stats["se"], 6)
wandb.summary["agents/accuracy_deff"]  = round(agent_acc_stats["deff"], 4)
wandb.summary["agents/accuracy_n_eff"] = round(agent_acc_stats["n_eff"], 2)
wandb.summary["agents/accuracy_icc"]   = round(agent_acc_stats["icc"], 4)

# Baseline accuracy
wandb.summary["baseline/accuracy"]       = round(baseline_acc_stats["p_hat"], 6)
wandb.summary["baseline/accuracy_ci"]    = [round(baseline_acc_stats["ci_lower"], 6),
                                             round(baseline_acc_stats["ci_upper"], 6)]
wandb.summary["baseline/accuracy_se"]    = round(baseline_acc_stats["se"], 6)
wandb.summary["baseline/accuracy_deff"]  = round(baseline_acc_stats["deff"], 4)
wandb.summary["baseline/accuracy_n_eff"] = round(baseline_acc_stats["n_eff"], 2)
wandb.summary["baseline/accuracy_icc"]   = round(baseline_acc_stats["icc"], 4)

# Agents execution time
_at = agent_time_stats
_ap = _at["percentiles"]
wandb.summary["agents/time_mean"]     = round(_at["mean"], 4)
wandb.summary["agents/time_std"]      = round(_at["std"], 4)
wandb.summary["agents/time_variance"] = round(_at["variance"], 4)
wandb.summary["agents/time_cv"]       = round(_at["cv"], 4)
wandb.summary["agents/time_median"]   = round(_at["median"], 4)
wandb.summary["agents/time_p25"]      = round(_ap["p25"], 4)
wandb.summary["agents/time_p75"]      = round(_ap["p75"], 4)
wandb.summary["agents/time_p95"]      = round(_ap["p95"], 4)
wandb.summary["agents/time_iqr"]      = round(_ap["p75"] - _ap["p25"], 4)
wandb.summary["agents/time_bca_ci"]   = [round(_at["ci_lower"], 4), round(_at["ci_upper"], 4)]
wandb.summary["agents/time_n_total"]  = _at["n_total"]

# Baseline execution time
_bt = baseline_time_stats
_bp = _bt["percentiles"]
wandb.summary["baseline/time_mean"]     = round(_bt["mean"], 4)
wandb.summary["baseline/time_std"]      = round(_bt["std"], 4)
wandb.summary["baseline/time_variance"] = round(_bt["variance"], 4)
wandb.summary["baseline/time_cv"]       = round(_bt["cv"], 4)
wandb.summary["baseline/time_median"]   = round(_bt["median"], 4)
wandb.summary["baseline/time_p25"]      = round(_bp["p25"], 4)
wandb.summary["baseline/time_p75"]      = round(_bp["p75"], 4)
wandb.summary["baseline/time_p95"]      = round(_bp["p95"], 4)
wandb.summary["baseline/time_iqr"]      = round(_bp["p75"] - _bp["p25"], 4)
wandb.summary["baseline/time_bca_ci"]   = [round(_bt["ci_lower"], 4), round(_bt["ci_upper"], 4)]
wandb.summary["baseline/time_n_total"]  = _bt["n_total"]

# # Time comparison (Welch t-test: agent vs baseline)
# wandb.summary["time_comparison/mean_agent"]        = round(time_comparison["mean_a"], 4)
# wandb.summary["time_comparison/mean_baseline"]     = round(time_comparison["mean_b"], 4)
# wandb.summary["time_comparison/mean_diff"]         = round(time_comparison["mean_diff"], 4)
# wandb.summary["time_comparison/t_statistic"]       = round(time_comparison["t_statistic"], 4)
# wandb.summary["time_comparison/df"]                = round(time_comparison["df"], 2)
# wandb.summary["time_comparison/p_value"]           = round(time_comparison["p_value"], 6)
# wandb.summary["time_comparison/significant_at_05"] = time_comparison["significant_at_05"]

# ── Upload correctness & time matrices as W&B artifact ───────
artifact = wandb.Artifact(
    f"correctness-matrices-{wandb.run.id}",
    type="evaluation-data",
    description="Correctness matrices (N x m) for agent and baseline, plus time matrices. GSM8K evaluation.",
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(f.name,
        agent_correctness=agent_correctness_matrix,
        baseline_correctness=baseline_correctness_matrix,
        agent_times=agent_time_matrix,
        baseline_times=baseline_time_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)
print("✅ Correctness & time matrices uploaded as W&B artifact.")

finish_wandb()
print("W&B run finished. View your dashboard at the URL above.")


  Agent Accuracy (GSM8K)
  Accuracy (pooled)            60.00%
  95% Wilson CI              [40.64, 76.67]%
  Cluster-robust SE           9.8527%
  Design Effect (DEFF)         2.427
  ICC                          0.713
  Effective sample size         24.7  (of 60 total)
  Problems × Seeds           20 × 3


  Baseline Accuracy (GSM8K)
  Accuracy (pooled)            48.33%
  95% Wilson CI              [35.15, 61.75]%
  Cluster-robust SE           7.0400%
  Design Effect (DEFF)         1.191
  ICC                          0.095
  Effective sample size         50.4  (of 60 total)
  Problems × Seeds           20 × 3


  Agent Execution Time (GSM8K)
  Mean (s)                     72.3464
  Std (s)                      30.6725
  Variance (s²)               940.8013
  CV                            0.4240
  95% BCa CI (s)             [65.9459, 81.7650]
  ── Percentiles ──
  Median / P50 (s)             63.7113
  P25 (s)                      54.2456
  P75 (s)                      77.2881
  IQ

wandb: uploading artifact run-ycuonn19-seed_456results_comparison-YLPGiQ; updating run metadata; uploading artifact correctness-matrices-ycuonn19
wandb: uploading artifact run-ycuonn19-seed_456results_comparison-YLPGiQ; uploading artifact correctness-matrices-ycuonn19
wandb: uploading artifact run-ycuonn19-seed_456results_comparison-YLPGiQ; uploading artifact correctness-matrices-ycuonn19; uploading history steps 359-362, summary, console lines 1346-1413
wandb: uploading artifact run-ycuonn19-seed_456results_comparison-YLPGiQ; uploading artifact correctness-matrices-ycuonn19
wandb: uploading wandb-summary.json; uploading config.yaml; uploading media/table/seed_456/results_comparison_362_48ae71f3259e1bd4950b.table.json; uploading output.log
wandb: 
wandb: Run history:
wandb:         gsm8k_sample/agents_accuracy ▁█▁▁██▁██▁██▁▁▁▁▁██▁█████▁▁▁█▁█▁███▁███▁
wandb:      gsm8k_sample/baseline/latency_s ▂▂▂▃▄▂▅▃▄▁▁▂▁▂▃▄▄▇▂▂▃▁▂▃▄▂█▅▂▁▆▂▂▃▃▂▃▂▁▂
wandb:   gsm8k_sample/baseline/output_chars ▂▃▃▁▂▂▃▂

W&B run finished. View your dashboard at the URL above.


## 13. Performance Analysis & Next Steps

### What W&B gives you

Open the W&B run link to explore:

- **Latency panel** — per-agent and total pipeline latency across tasks, showing where time is spent.
- **Results table** — side-by-side text comparison of all four outputs (proposer, critic, refiner, baseline) for every task.
- **Overhead ratio** — how much more expensive the 3-agent pipeline is vs. the single-shot baseline.

### Architecture improvements enabled by LangChain

The LangChain refactor opens the door to several extensions without rewriting the core loop:

- **Multi-turn debate** — wrap the Proposer → Critic step in a loop with a LangChain `SequentialChain` or custom `RunnableSequence`.
- **Dynamic routing** — use a LangChain `RouterChain` to skip the Critic if the Proposer's confidence is high.
- **Tool-augmented agents** — give the Critic access to a calculator tool via `AgentExecutor` to verify arithmetic.
- **Model swaps** — replace the `Mistral-7B-Instruct-v2` for newer SOTA SLMs.

### Suggested next steps

1. **Hyperparameter sweeps** — use `wandb.sweep()` over temperature, top_p, and max_new_tokens.
2. **Model comparison** — run the same pipeline with Llama-3-8B or Gemma-7B and compare W&B dashboards.
3. **Human evaluation** — log outputs to a W&B `Table` with a "human_score" column and annotate post-hoc.
